# fine-tuning using your in-house assay

This tutorial is for users who have already run a functional assay and would like to answer the question "what if I had run my assay with other drugs, other CRISPR treatments, or in other cell lines?"

For example, [Tieu et al (2024)](https://doi.org/10.1016/j.cell.2024.01.035) run a combinatorial CAR-T transduction with 24 guides for a total of 576 pairwise combinations. We can fine-tune Prophet on this dataset and make predictions for genes spanning the entire genome and additional combinations therein.

In [ ]:
import pandas as pd
import yaml
from prophet import Prophet, set_config
from prophet.data import universal_processing

We load in a config file to get the file paths for the embeddings which should be used. These can be downloaded from the links on Github.

In [2]:
with open("config_file_finetuning.yaml", "r") as f:
    config = set_config(yaml.safe_load(f))

We've randomly select one of the pretrained model checkpoints (noted in the config file above) to train on here. For more robust results, we recommend training with several checkpoints and taking the ensemble prediction.

In [3]:
config.ckpt_path

'/lustre/groups/ml01/projects/super_rad_project/pretrained_prophet/everything/iv_0_iv_out_multitest_300cl_1219iv_512model_8layers_Falsesimpler_Truemask_0.0001lr_Falseexplicitphenotype_10000warmup_150001max_iters_Falseunbalanced_0.01wd_4096bs_Falseft/iv_0_seed_2024/epoch=29.ckpt'

Load in the pretrained model.

In [4]:
model = Prophet(
    iv_emb_path=config.genes_prior,
    cl_emb_path=config.cell_lines_prior,
    ph_emb_path=None,
    model_pth=config.ckpt_path,
)

returning trained model!
Gene net:  Sequential(
  (0): Linear(in_features=1219, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Cell line net:  Sequential(
  (0): Linear(in_features=300, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Regressor:  Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
  (4): GELU(approximate='none')
  (5): Linear(in_features=512, out_features=1, bias=True)
)


Here we've provided two examples for finetuning to demonstrate the variety of datasets on which you can finetune Prophet. We recommend that all datasets have values minmaxed to between 0 and 1 before finetuning.

### GDSC2

GDSC2 (https://www.cancerrxgene.org/) is a cancer-screening dataset with titrated IC50s.

 - cell states: cancer cell lines
 - interventions: small molecule singletons
 - readout: IC50 of viability as measured using CellTitreGlo at 72hrs, fitted over multiple concentrations

In [5]:
gdsc_data_path = (
    "/lustre/groups/ml01/projects/super_rad_project/data/GDSC_notscaled_minmax.csv"
)
data_label = pd.read_csv(gdsc_data_path, index_col=0)

data_label["iv2"] = (
    "negative_drug"  # there is no second compound so we specify negative_drug so the token is masked
)
data_label["phenotype"] = (
    "GDSC"  # this is the label to use for this readout when you run inference later
)

data_label = universal_processing(data_label)
data_label.head()

,cell_line,iv1,value,iv_name,phenotype,iv2
0,PFSK1,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,0.323078,Camptothecin,GDSC,negative_drug
1,A673,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,0.172422,Camptothecin,GDSC,negative_drug
2,ES5,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,0.239133,Camptothecin,GDSC,negative_drug
3,ES7,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,0.164659,Camptothecin,GDSC,negative_drug
4,EW11,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,0.222290,Camptothecin,GDSC,negative_drug


In [6]:
config.dirpath = "./ckpts_GDSC/"  # set the save directory here; you can also set it in the config file
model.train(
    data_label,
    iv_col=["iv1", "iv2"],
    cl_col="cell_line",
    ph_col="phenotype",
    model_config=config,
)

Removing 61 such as ['123138', '123829', '150412', '50869', '615590'] from ['iv1', 'iv2']. 393646 rows remaining.
Removing 285 such as ['451LU', '7860', 'ALLPO', 'ARH77', 'ATN1'] from ['cell_line']. 280202 rows remaining.
Fitting model.
pytorch model, finetuning
Gene net:  Sequential(
  (0): Linear(in_features=1219, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Cell line net:  Sequential(
  (0): Linear(in_features=300, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Regressor:  Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
  (4): GELU(approximate='none')
  (5): Linear(in_features=512, out_features=1, bias=True)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


R2 average:  False
Running with early stopping: True
Early stopping patience: 20


You are using a CUDA device ('NVIDIA A100-PCIE-40GB MIG 3g.20gb') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-c51c82f2-7e04-56a8-8b74-4211c9821715]

  | Name                | Type               | Params
-----------------------------------------------------------
0 | learnable_embedding | Embedding          | 512 K 
1 | embedding_dropout   | Dropout            | 0     
2 | gene_net            | Sequential         | 887 K 
3 | drug_net            | Sequential         | 887 K 
4 | cl_net              | Sequential         | 416 K 
5 | transformer         | TransformerEncoder | 16.8 M
6 | output_net          | Sequential         | 525 K 
----------------------------------------------------

Epoch 0: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████| 124/124 [00:37<00:00,  3.26it/s, v_num=8, train_loss=0.119]
Validation: |                                                                                                                                                   | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 124/124 [00:38<00:00,  3.25it/s, v_num=8, train_loss=0.0325]
Validation: |                                                                                                                                                   | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 124/124 [00:40<00:00,  3.10it/s, v_num=8, train_loss=0.0325]

`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 124/124 [00:41<00:00,  3.00it/s, v_num=8, train_loss=0.0325]


### TieuQi

This is a T-cell proliferation dataset from https://doi.org/10.1016/j.cell.2024.01.035.

- cell state: T-cell proliferation
- intervention: combinatorial CRISPRi
- readout: Log2FC of CD8+ T-cells vs. control pDNA

In [7]:
tieu_data_path = "/ictstr01/home/icb/yuge.ji/projects/super_rad_project/data/TieuQi_lfc_day11_minmax.csv"
data_label = pd.read_csv(tieu_data_path, index_col=0)

data_label["phenotype"] = (
    "T-cell_viability"  # this is the label to use for this readout when you run inference later
)

data_label = universal_processing(data_label)
data_label

,iv1,iv2,value,cell_line,phenotype
0,batf3,batf3,0.633443,JURKAT,T-cell_viability
1,batf3,cblb,0.544929,JURKAT,T-cell_viability
2,batf3,ctla4,0.483662,JURKAT,T-cell_viability
3,batf3,dhx37,0.693173,JURKAT,T-cell_viability
4,batf3,fas,0.594995,JURKAT,T-cell_viability
...,...,...,...,...,...
1245,tigit,zc3h12a,0.604678,JURKAT,T-cell_viability
1246,tox,zc3h12a,0.596647,JURKAT,T-cell_viability
1247,tox2,zc3h12a,0.418890,JURKAT,T-cell_viability
1248,trac,zc3h12a,0.625894,JURKAT,T-cell_viability


In [8]:
config.dirpath = "./ckpts_TieuQi/"
model.train(
    data_label,
    iv_col=["iv1", "iv2"],
    cl_col="cell_line",
    ph_col="phenotype",
    model_config=config,
)

Removing 4 such as ['tceb2', 'tet2', 'tigit', 'trac'] from ['iv1', 'iv2']. 861 rows remaining.
Removing 0 such as [] from ['cell_line']. 861 rows remaining.
Fitting model.
pytorch model, finetuning
Gene net:  Sequential(
  (0): Linear(in_features=1219, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Cell line net:  Sequential(
  (0): Linear(in_features=300, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Regressor:  Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
  (4): GELU(approximate='none')
  (5): Linear(in_features=512, out_features=1, bias=True)
)


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-c51c82f2-7e04-56a8-8b74-4211c9821715]

  | Name                | Type               | Params
-----------------------------------------------------------
0 | learnable_embedding | Embedding          | 512 K 
1 | embedding_dropout   | Dropout            | 0     
2 | gene_net            | Sequential         | 887 K 
3 | drug_net            | Sequential         | 887 K 
4 | cl_net              | Sequential         | 416 K 
5 | transformer         | TransformerEncoder | 16.8 M
6 | output_net          | Sequential         | 525 K 
-----------------------------------------------------------
20.1 M    Trainable params
0         Non-trainable params
20.1 M    Total params
80.206    Total estimated model params size (MB)


R2 average:  False
Running with early stopping: True
Early stopping patience: 20
Epoch 0: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.07it/s, v_num=9, train_loss=0.638]
Validation: |                                                                                                                                                   | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  1.01it/s, v_num=9, train_loss=0.592]
Validation: |                                                                                                                                                   | 0/? [00:00<?, ?it/s]
Epoch 1: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  0.65it/s, v_num=9, train_loss=0.592]

`Trainer.fit` stopped: `max_epochs=2` reached.


Epoch 1: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:03<00:00,  0.31it/s, v_num=9, train_loss=0.592]


## Loading in a model checkpoint

In [9]:
pretrained_checkpoint_path = "./ckpts_TieuQi/epoch=0-step=1.ckpt"
model = Prophet(
    iv_emb_path=config.genes_prior,
    cl_emb_path=config.cell_lines_prior,
    ph_emb_path=None,
    model_pth=pretrained_checkpoint_path,
)

returning trained model!
Gene net:  Sequential(
  (0): Linear(in_features=1219, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Cell line net:  Sequential(
  (0): Linear(in_features=300, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.1, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
)
Regressor:  Sequential(
  (0): Linear(in_features=512, out_features=512, bias=True)
  (1): GELU(approximate='none')
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=512, out_features=512, bias=True)
  (4): GELU(approximate='none')
  (5): Linear(in_features=512, out_features=1, bias=True)
)


As you can see, this checkpoint can now be loaded for inference! See other notebooks in `tutorials` for how to perform inference, or copy this notebook for more finetuning.